# 04a. Unadjusted Outcomes Overview


Short summary:
[04a - Unadjusted Outcomes Overview (Non-ICU Admissions)](../docs/04a_unadjusted_outcomes_SHORT.md)

## 1. Purpose

This notebook provides a descriptive overview of unadjusted (crude) in-hospital outcomes stratified by early RAAS inhibitor exposure among non-ICU adult hospital admissions.

The purpose of this step is to summarize raw outcome distributions prior to any multivariable adjustment, effect estimation, or causal modeling.

## 2. Background

Early RAAS inhibitor use has been examined in prior observational studies of hospitalized populations, with reported associations with in-hospital outcomes.

Before conducting adjusted analyses, it is informative to examine unadjusted outcome patterns to understand the raw distribution of events, exposure prevalence, and baseline differences between groups.

## 3. Methods

### 3.1 Data Source and Study Population

- **MIMIC-IV v3.1** (BigQuery public dataset)
- Project: `mimic-iv-portfolio`

**Source Tables:**
  - `mimic-iv-portfolio.nonicu_raas.analysis_dataset`<br>
    (created in [02_exposure.ipynb](02_exposure.ipynb) using [03_build_analysis_dataset.sql](../sql/03_build_analysis_dataset.sql))

This retrospective observational study used data from the Medical Information Mart for Intensive Care IV (MIMIC-IV), a large, publicly available, de-identified clinical database containing detailed electronic health record data from hospital admissions at Beth Israel Deaconess Medical Center.

The study population consisted of adult patients (aged ≥ 18 years) with at least one recorded hospital admission. Admissions with missing key identifiers or implausible demographic values were excluded. Each hospital admission is a descriptive observation; a patient may contribute multiple admissions. This notebook does not estimate within-patient uncertainty; 04b separately adds patient-cluster intervals and patient-disjoint retrospective discrimination. Patient age at admission was derived from anchor age information provided by MIMIC-IV.

Because the database is fully de-identified, institutional review board approval and informed consent were not required.

### 3.2 Exposure and Outcome Definitions

The exposure of interest was early use of renin–angiotensin–aldosterone system (RAAS) inhibitors, defined using inpatient medication prescription records indicating use of either an angiotensin-converting enzyme inhibitor (ACE inhibitor) or an angiotensin receptor blocker (ARB) during the early phase of hospitalization, as specified by pre-defined, time-restricted criteria applied upstream.

The binary indicator `raas_any_early` denotes an ACE inhibitor or ARB prescription record with `admittime <= starttime < admittime + 24 hours`. It does not confirm administration, adherence or outpatient chronic use. A zero flag means no qualifying early record, not proven absence of medication use.

The outcome was in-hospital mortality, defined using the hospital expiration flag in MIMIC-IV. Mortality during the index hospitalization was coded as 1, and survival to discharge as 0.


### 3.3 Covariates

Baseline demographic and admission-related variables were summarized for descriptive purposes only. These included:

- Age at admission
- Sex
- Race and ethnicity
- Admission type
- Insurance category

No covariates were used for statistical adjustment in this analysis.

### 3.4 Statistical Analysis

This analysis was descriptive in nature.
Unadjusted outcome summaries were computed separately for admissions with and without early RAAS inhibitor exposure.

Specifically, we reported:

- The number of admissions in each exposure group
- The number of in-hospital deaths
- Crude in-hospital mortality rates
- Length of stay summaries by exposure group

No regression modeling, hypothesis testing, or multivariable adjustment was performed at this stage.

### 3.5 Interpretation Framework

All results presented in this analysis represent unadjusted comparisons and are intended to provide descriptive context only.

Differences in outcomes between exposure groups may reflect confounding by age, admission characteristics, baseline clinical severity, or other unmeasured factors. Accordingly, no causal interpretation was made based on these findings.

Adjusted analyses and absolute effect estimation were conducted in subsequent modeling steps.


## 4. Data Preparation
### 4.1 Dataset Loading and Initial Inspection

## Reading the results

Data sources and execution mode are summarized once below, after the destination settings. Expand **Calculation inputs and source definitions** to trace individual SQL definitions and aggregate references.

Each result is generated by Python in this order: **number/title → figure or aggregate table → how to read → computed result summary → interpretation limits**. Cohort definitions, exposures, adjustment variables and estimators are unchanged. See the [reproduction instructions](../REPRODUCIBILITY.md#python-generated-notebook-results).


The input-definition index describes the Python objects/queries used in calculation. The separate **Saved aggregates and verification files (CSV/JSON)** list identifies saved results and checking records, rather than original MIMIC-IV source tables. Live notebook displays use in-memory results; saved-output replay and validation workflows may read these files.

In [ ]:
# Presentation uses only objects produced by this notebook execution.
from pathlib import Path as _ResultPath
import sys as _result_sys
_result_root = _ResultPath.cwd().resolve()
if _result_root.name == "notebooks":
    _result_root = _result_root.parent
if not (_result_root / "scripts" / "nonicu_notebook_results.py").is_file():
    raise RuntimeError("Run this notebook from the repository root or its notebooks directory")
if str(_result_root / "scripts") not in _result_sys.path:
    _result_sys.path.insert(0, str(_result_root / "scripts"))
from nonicu_notebook_results import NotebookResults
notebook_results = NotebookResults(globals())


In [ ]:
from google.cloud import bigquery
from google.auth import default

PROJECT_ID = "mimic-iv-portfolio"
DATASET = "nonicu_raas"

TABLE_ANALYSIS = f"{PROJECT_ID}.{DATASET}.analysis_dataset"  # Created in 03_build_analysis_dataset.sql

creds, _ = default()
client = bigquery.Client(project=PROJECT_ID, credentials=creds)

In [ ]:
# Shared source references and execution mode for all numbered results.
notebook_results.overview(['Table 04a.1', 'Table 04a.2', 'Table 04a.3', 'Table 04a.4', 'Table 04a.5', 'Table 04a.6', 'Table 04a.7', 'Figure 04a.1', 'Table 04a.8'])


In [ ]:
# Helper for read-only SELECT queries → DataFrame
def query_to_df(query) :
    """
    Run a SELECT query in BigQuery and return a pandas DataFrame.
    """
    job = client.query(query)
    return job.to_dataframe(create_bqstorage_client=False)

In [ ]:
with notebook_results.capture('Table 04a.1', globals()):
    # Sanity check of the analysis dataset
    q = f"""
SELECT *
FROM `{TABLE_ANALYSIS}`
"""
    df = query_to_df(q)

    df.info()
    print("Shape:", df.shape)
    _notebook_result_value = (df["raas_any_early"].value_counts(dropna=False))


### 4.2 Dataset Overview and Descriptive Checks

In [ ]:
with notebook_results.capture('Table 04a.2', globals()):
    df["raas_any_early"].value_counts(dropna=False)
    _notebook_result_value = (df["hospital_expire_flag"].value_counts(dropna=False))


In [ ]:
with notebook_results.capture('Table 04a.3', globals()):
    print("N admissions:", df.shape[0])
    print(f"Overall mortality rate: {(df['hospital_expire_flag'].mean()*100):.2f} %")
    _notebook_result_value = (print("Median age:", df["age"].median()))


### 4.3 Outcome and Exposure Construction

In [ ]:
with notebook_results.capture('Table 04a.4', globals()):
    expo_summary = (
        df["raas_any_early"]
        .map({1: "Early RAAS inhibitor use", 0: "No early RAAS inhibitor use"})
        .value_counts(dropna=False)
        .rename_axis("Exposure group")
        .reset_index(name="Number of admissions")
    )

    _notebook_result_value = (expo_summary)


### 4.4 Baseline Age Distribution

In [ ]:
with notebook_results.capture('Table 04a.5', globals()):
    _notebook_result_value = (df.groupby("raas_any_early")["age"].describe())


## 5. Outputs
### 5.1 Unadjusted Outcomes

In [ ]:
with notebook_results.capture('Table 04a.6', globals()):
    tbl = (
        df.groupby("raas_any_early")
          .agg(
              n_admissions=("raas_any_early", "count"),
              n_events=("hospital_expire_flag", "sum"),
              mortality_rate=("hospital_expire_flag", "mean"),
          )
    )
    _notebook_result_value = (tbl)


**Aggregate Python Export**

Selected unadjusted summaries are exported to `python/outputs/unadjusted_outcomes.csv` using the counts above. This does not change the analysis.


In [ ]:
with notebook_results.capture('Table 04a.7', globals()):
    # Export selected unadjusted outcome summaries for downstream validation
    from pathlib import Path

    notebooks_dir = Path.cwd() if Path.cwd().name == "notebooks" else Path.cwd() / "notebooks"
    project_root = notebooks_dir.parent
    output_path = project_root / "python" / "outputs" / "unadjusted_outcomes.csv"
    output_path.parent.mkdir(parents=True, exist_ok=True)

    python_unadjusted_outcomes = tbl.reset_index().rename(columns={
        "raas_any_early": "exposure",
        "n_admissions": "n",
        "n_events": "deaths",
        "mortality_rate": "mortality_proportion",
    })
    python_unadjusted_outcomes["exposure_group"] = python_unadjusted_outcomes["exposure"].map({
        0: "No early RAAS inhibitor use",
        1: "Early RAAS inhibitor use",
    })
    python_unadjusted_outcomes["non_deaths"] = (
        python_unadjusted_outcomes["n"] - python_unadjusted_outcomes["deaths"]
    )
    python_unadjusted_outcomes["crude_odds"] = (
        python_unadjusted_outcomes["deaths"] / python_unadjusted_outcomes["non_deaths"]
    )

    reference_odds = python_unadjusted_outcomes.loc[
        python_unadjusted_outcomes["exposure"].eq(0),
        "crude_odds",
    ].iloc[0]
    python_unadjusted_outcomes["crude_odds_ratio_vs_no_early_raas"] = (
        python_unadjusted_outcomes["crude_odds"] / reference_odds
    )

    python_unadjusted_outcomes = python_unadjusted_outcomes[
        [
            "exposure",
            "exposure_group",
            "n",
            "deaths",
            "non_deaths",
            "mortality_proportion",
            "crude_odds",
            "crude_odds_ratio_vs_no_early_raas",
        ]
    ]

    python_unadjusted_outcomes.to_csv(output_path, index=False)

    _notebook_result_value = (python_unadjusted_outcomes)


**README Figure Export**

The unadjusted mortality figure is saved under `assets/` for the portfolio README. This figure is descriptive only and does not change the clinical analysis logic.


In [ ]:
with notebook_results.capture('Figure 04a.1', globals()):
    # Save README figure: unadjusted in-hospital mortality by early RAAS exposure
    import matplotlib.pyplot as plt

    assets_dir = project_root / "assets"
    assets_dir.mkdir(parents=True, exist_ok=True)
    figure_path = assets_dir / "unadjusted_mortality_by_raas.png"

    plot_df = python_unadjusted_outcomes.copy()
    plot_df["mortality_percent"] = plot_df["mortality_proportion"] * 100
    plot_df = plot_df.sort_values("exposure")
    # Display labels describe prescription-record exposure; retain aggregate CSV values.
    plot_df["exposure_label"] = plot_df["exposure"].map({
        0: "No documented early\nRAAS prescription exposure",
        1: "Documented early\nRAAS prescription exposure",
    })

    fig, ax = plt.subplots(figsize=(8, 4.8))
    bars = ax.bar(
        plot_df["exposure_label"],
        plot_df["mortality_percent"],
        color=["#94a3b8", "#2563eb"],
    )
    ax.set_ylabel("In-hospital mortality (%)")
    ax.set_title("Unadjusted mortality by documented early RAAS prescription exposure")
    ax.set_ylim(0, max(plot_df["mortality_percent"]) * 1.35)
    ax.spines[["top", "right"]].set_visible(False)
    ax.tick_params(axis="x", rotation=0)

    for bar, (_, row) in zip(bars, plot_df.iterrows()):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height(),
            f'{row["mortality_percent"]:.2f}%\n{int(row["deaths"]):,}/{int(row["n"]):,}',
            ha="center",
            va="bottom",
            fontsize=10,
        )

    fig.text(
        0.01,
        0.01,
        "Prescription records do not confirm administration.\nDescriptive, unadjusted comparison; observational association only.",
        fontsize=8.5,
        color="#475569",
    )
    fig.tight_layout(rect=[0, 0.10, 1, 1])
    fig.savefig(figure_path, dpi=200, bbox_inches="tight")
    plt.show()

    _notebook_result_value = (figure_path.relative_to(project_root))


### 5.2 Unadjusted Length of Stay by Early RAAS Exposure

In [ ]:
import numpy as np
df["expo_group"] = np.where(df["raas_any_early"] == 1, "RAAS early", "No RAAS early")

In [ ]:
with notebook_results.capture('Table 04a.8', globals()):
    los_summary = (
        df
        .groupby("expo_group")["hosp_los"]
        .agg(["median", "mean"])
        .round(2)
        .reset_index()
    )

    _notebook_result_value = (los_summary)


## Interpretation boundary

Use the generated result summaries and fit-status diagnostics above. These are observational prescription-record contrasts, not causal treatment effects. Repeated admissions, estimator-specific uncertainty and fixed-time prediction cohorts must be distinguished. Whole-admission LOS is descriptive. Age-specific mean-profile curves are not an exposure-by-age interaction test.

See the [historical clinical execution evidence](../docs/CLINICAL_REEXECUTION_615de01.md) and [reproducible scope](../docs/REPRODUCIBLE_SCOPE.md). The separate 04c prediction-time analysis retains later ICU transfers and must not inherit the retrospective-discrimination AUCs.
